In [0]:
df_employees = spark.sql(f"select * from regis_healthcare.silver.employees;")
df_employees.createOrReplaceTempView("employees")

In [0]:
# 3. Dim_Employee --> Source: employees
# | Column          |
# | --------------- |
# | employee_key    |
# | employee_id     |
# | first_name      |
# | last_name       |
# | job_title       |
# | employment_type |
# | status          |
# | salary          |
# | hire_date       |


Dim_Employee = spark.sql("""select * from employees order by employee_id""")
# display(Dim_Employee)

from pyspark.sql.functions import col, regexp_replace
Dim_Employee = Dim_Employee.withColumn(
    "employee_key",
    regexp_replace(col("employee_id"), "^EMP", "").cast("int")
)
# from pyspark.sql.functions import col, date_format
# # Create date_key column in YYYYMMDD format
# Dim_Employee = Dim_Employee.withColumn("emp_hire_date_key", date_format(col("hire_date"), "yyyyMMdd"))
# # Optionally cast to integer for warehouse-style keys
# Dim_Employee = Dim_Employee.withColumn("emp_hire_date_key", col("emp_hire_date_key").cast("int"))
# display(df_employees)
Dim_Employee = Dim_Employee.select(
    "employee_key",
    "employee_id",
    "first_name",
    "last_name",
    "job_title",
    "employment_type",
    "status",
    "salary",
    "hire_date")
display(Dim_Employee)

#### cataloge 

In [0]:
# Dim_Employee.write\
#     .format("delta")\
#     .option("mergeSchema","true")\
#     .option("overwriteSchema","true")\
#         .option("delta.enableChangeDataFeed","true")\
#             .mode("overwrite")\
# .saveAsTable(f"regis_healthcare.gold.dim_employee")

In [0]:
Dim_Employee.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.sb_dim_employee")
print(Dim_Employee.count())

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Load Delta table with correct fully-qualified name
delta_table = DeltaTable.forName(spark, "regis_healthcare.gold.dim_employee")
# Create DataFrame from source table with correct fully-qualified name
# sb_dim_products
df_child_products = (
    spark.table("regis_healthcare.gold.sb_dim_employee")
    .select("*")
)
# Perform merge
delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.employee_key = source.employee_key"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
dim_df = spark.sql(f"select * from regis_healthcare.gold.dim_employee;")
print(dim_df.count())

sb_dim_df = spark.sql(f"select * from regis_healthcare.gold.sb_dim_employee;")
print(sb_dim_df.count())

#### s3 loading

In [0]:
# # gold load to s3
# Dim_Employee.write\
#     .format("delta")\
#     .option("mergeSchema","true")\
#     .option("overwriteSchema","true")\
#         .option("delta.enableChangeDataFeed","true")\
#             .mode("overwrite")\
# .save(f"s3://regis-healthcare/gold-delta-table/dim_employee")

In [0]:
from delta.tables import DeltaTable

# ✅ Path to your Delta table stored in S3
delta_table_path = f"s3://regis-healthcare/gold-delta-table/dim_employee"

# ✅ Load target Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)

# ✅ Source DataFrame (example: df_child_products)
source_df = Dim_Employee

# ✅ Perform MERGE with upsert logic
(
    delta_table.alias("target")
    .merge(
        source_df.alias("source"),
        "target.employee_key = source.employee_key"
    )
    .whenMatchedUpdateAll()      # Update all columns when matched
    .whenNotMatchedInsertAll()   # Insert all columns when not matched
    .execute()
)
